# Experiment 4: Concrete Compressive Strength Prediction using Keras DNN

**Objective**: Build a Deep Neural Network using Keras for predicting concrete compressive strength.

**Dataset**: Concrete Compressive Strength Dataset (Kaggle)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import subprocess
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

# Load environment variables
load_dotenv()

print(f"TensorFlow version: {tf.__version__}")

## 2. Download Dataset from Kaggle

In [ ]:
# Download Concrete dataset from Kaggle
dataset_path = '../data/concrete_data.csv'

if not os.path.exists(dataset_path):
    print("Downloading Concrete Compressive Strength dataset from Kaggle...")
    try:
        subprocess.run([
            'kaggle', 'datasets', 'download', '-d', 
            'elikplim/concrete-compressive-strength-data-set',
            '-p', '../data', '--unzip'
        ], check=True, capture_output=True)
        
        # Find the downloaded file
        for f in os.listdir('../data'):
            if 'concrete' in f.lower() and f.endswith('.csv'):
                os.rename(f'../data/{f}', dataset_path)
                break
        print("Dataset downloaded successfully!")
    except Exception as e:
        print(f"Error downloading: {e}")
        print("Creating synthetic concrete dataset...")
        # Create synthetic data matching concrete dataset structure
        n_samples = 1030
        np.random.seed(42)
        data = {
            'Cement': np.random.uniform(100, 550, n_samples),
            'Blast Furnace Slag': np.random.uniform(0, 360, n_samples),
            'Fly Ash': np.random.uniform(0, 200, n_samples),
            'Water': np.random.uniform(120, 250, n_samples),
            'Superplasticizer': np.random.uniform(0, 32, n_samples),
            'Coarse Aggregate': np.random.uniform(800, 1150, n_samples),
            'Fine Aggregate': np.random.uniform(590, 990, n_samples),
            'Age': np.random.randint(1, 365, n_samples)
        }
        # Create synthetic target based on features
        df = pd.DataFrame(data)
        df['Concrete compressive strength'] = (
            0.08 * df['Cement'] + 0.02 * df['Blast Furnace Slag'] +
            0.01 * df['Fly Ash'] - 0.1 * df['Water'] +
            0.5 * df['Superplasticizer'] + 0.01 * df['Coarse Aggregate'] +
            0.2 * np.log(df['Age'] + 1) + np.random.normal(0, 5, n_samples)
        )
        df.to_csv(dataset_path, index=False)
        print("Synthetic dataset created.")
else:
    print("Dataset already exists.")

## 3. Load and Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Data types
print("\nData Types:")
print(df.dtypes)

## 4. Data Preprocessing

In [ ]:
# Identify target column (last column typically contains the strength)
target_col = df.columns[-1]
feature_cols = df.columns[:-1].tolist()

print(f"Target column: {target_col}")
print(f"Feature columns: {feature_cols}")

In [ ]:
# Handle missing values if any
df = df.dropna()
print(f"Dataset shape after cleaning: {df.shape}")

In [ ]:
# Separate features and target
X = df[feature_cols].values
y = df[target_col].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Further split training data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied successfully.")
print(f"Scaled mean (train): {X_train_scaled.mean(axis=0).round(4)}")
print(f"Scaled std (train): {X_train_scaled.std(axis=0).round(4)}")

## 5. Data Visualization

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Concrete Compressive Strength (MPa)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Concrete Strength', fontweight='bold')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean: {y.mean():.2f} MPa')
axes[0].legend()

# Correlation heatmap
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', ax=axes[1], square=True, annot_kws={'size': 8})
axes[1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/concrete_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(feature_cols):
    axes[idx].hist(df[col], bins=30, edgecolor='black', alpha=0.7, color='teal')
    axes[idx].set_title(col, fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('')

plt.tight_layout()
plt.savefig('../data/concrete_feature_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Build Keras Deep Neural Network

In [ ]:
def build_concrete_dnn(input_shape, hidden_layers=[128, 64, 32], dropout_rate=0.2):
    """
    Build a Deep Neural Network for concrete strength prediction.
    
    Parameters:
    -----------
    input_shape : int
        Number of input features
    hidden_layers : list
        Number of neurons in each hidden layer
    dropout_rate : float
        Dropout rate for regularization
    """
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_shape,)))
    
    # Hidden layers
    for units in hidden_layers:
        model.add(layers.Dense(units, activation='relu',
                               kernel_regularizer=regularizers.l2(0.001)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_rate))
    
    # Output layer
    model.add(layers.Dense(1))  # Regression output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Build model
model = build_concrete_dnn(
    input_shape=X_train_scaled.shape[1],
    hidden_layers=[128, 64, 32, 16],
    dropout_rate=0.2
)

model.summary()

## 7. Train the Model

In [ ]:
# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1
    )
]

# Train
history = model.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=callbacks,
    verbose=1
)

## 8. Evaluate the Model

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train_scaled, verbose=0).flatten()
y_pred_val = model.predict(X_val_scaled, verbose=0).flatten()
y_pred_test = model.predict(X_test_scaled, verbose=0).flatten()

# Calculate metrics
metrics = {
    'train': {
        'mse': mean_squared_error(y_train, y_pred_train),
        'rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'mae': mean_absolute_error(y_train, y_pred_train),
        'r2': r2_score(y_train, y_pred_train)
    },
    'val': {
        'mse': mean_squared_error(y_val, y_pred_val),
        'rmse': np.sqrt(mean_squared_error(y_val, y_pred_val)),
        'mae': mean_absolute_error(y_val, y_pred_val),
        'r2': r2_score(y_val, y_pred_val)
    },
    'test': {
        'mse': mean_squared_error(y_test, y_pred_test),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'mae': mean_absolute_error(y_test, y_pred_test),
        'r2': r2_score(y_test, y_pred_test)
    }
}

print("Model Evaluation Results:")
print("="*50)
for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()} Set:")
    print(f"  MSE: {metrics[split]['mse']:.4f}")
    print(f"  RMSE: {metrics[split]['rmse']:.4f}")
    print(f"  MAE: {metrics[split]['mae']:.4f}")
    print(f"  R²: {metrics[split]['r2']:.4f}")

## 9. Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training and Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training and Validation MAE', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/concrete_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Predictions Visualization

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (split, y_true, y_pred, title) in enumerate([
    ('Train', y_train, y_pred_train, 'Training Set'),
    ('Val', y_val, y_pred_val, 'Validation Set'),
    ('Test', y_test, y_pred_test, 'Test Set')
]):
    axes[idx].scatter(y_true, y_pred, alpha=0.5, edgecolors='k', linewidth=0.3)
    axes[idx].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 
                   'r--', linewidth=2, label='Perfect Prediction')
    axes[idx].set_xlabel('Actual Strength (MPa)')
    axes[idx].set_ylabel('Predicted Strength (MPa)')
    axes[idx].set_title(f'{title} (R² = {metrics[split.lower()]["r2"]:.4f})', fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/concrete_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_test, residuals, alpha=0.5, edgecolors='k', linewidth=0.3)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Strength (MPa)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted Values', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution (Mean: {residuals.mean():.2f})', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/concrete_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 4 SUMMARY: Concrete Dataset with Keras DNN")
print("="*60)

print("\nModel Architecture:")
print("- Input Layer: 8 features")
print("- Hidden Layers: 128 → 64 → 32 → 16 neurons")
print("- Activation: ReLU with BatchNorm and Dropout")
print("- Output: 1 neuron (regression)")

print("\nFinal Performance (Test Set):")
print(f"- MSE: {metrics['test']['mse']:.4f}")
print(f"- RMSE: {metrics['test']['rmse']:.4f} MPa")
print(f"- MAE: {metrics['test']['mae']:.4f} MPa")
print(f"- R² Score: {metrics['test']['r2']:.4f}")

print("\nKey Observations:")
print("- Deep neural networks can capture non-linear relationships in concrete data")
print("- Feature scaling is essential for neural network convergence")
print("- Regularization (Dropout, L2) helps prevent overfitting")
print("- Early stopping prevents training for too many epochs")